In [1]:
import os
import cv2
import dlib
import imutils
import numpy as np
from imutils import face_utils

In [2]:
BASE_PATH = os.path.expanduser("~/100xDevs/personality/Personality-Classifier")

INPUT_BASE = os.path.join(
    BASE_PATH,
    "data/processed/double_chin_dataset"
)[:100]

OUTPUT_BASE = os.path.join(
    BASE_PATH,
    "data/processed/jaw_dataset"
)

PREDICTOR_PATH = os.path.join(
    BASE_PATH,
    "models/landmark_model/shape_predictor_68_face_landmarks.dat"
)

print("Input:", INPUT_BASE)
print("Output:", OUTPUT_BASE)

Input: /home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/double_chin_dataset
Output: /home/yeezy/100xDevs/personality/Personality-Classifier/data/processed/jaw_dataset


In [3]:
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(PREDICTOR_PATH)

print("Dlib loaded successfully.")

Dlib loaded successfully.


In [4]:
def extract_jaw(image):
    image = imutils.resize(image, width=600)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    rects = detector(gray, 1)

    if len(rects) != 1:
        return None

    for rect in rects:
        shape = predictor(gray, rect)
        shape = face_utils.shape_to_np(shape)

        start, end = 0, 17  # Jaw landmarks

        (x, y, w, h) = cv2.boundingRect(np.array([shape[start:end]]))
        roi = image[y:y + h + 50, x:x + w]

        try:
            roi = imutils.resize(roi, width=250)
            return roi
        except:
            return None

    return None

In [5]:
def process_split(split,max_images=None):
    for label in ["double_chin", "no_double_chin"]:

        input_folder = os.path.join(INPUT_BASE, split, label)
        output_folder = os.path.join(OUTPUT_BASE, split, label)

        os.makedirs(output_folder, exist_ok=True)

        processed = 0
        skipped = 0

        for file in os.listdir(input_folder):

            if not file.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            input_path = os.path.join(input_folder, file)
            output_path = os.path.join(output_folder, file)

            image = cv2.imread(input_path)

            if image is None:
                skipped += 1
                continue

            roi = extract_jaw(image)

            if roi is not None:
                cv2.imwrite(output_path, roi)
                processed += 1
            else:
                skipped += 1

        print(f"{split} - {label} -> Processed: {processed}, Skipped: {skipped}")

In [ ]:
process_split("train")
process_split("valid")

print("\n✅ Jaw dataset creation complete.")

valid - double_chin -> Processed: 1859, Skipped: 33
